# 01 · Los datos reales: cuatro estudios clínicos de supervivencia

**S.Y.N.A.P.S.E.** · Investigadora principal: Maricel Meneses Gómez · Colaborador: Yoandy Ramírez Delgado

Antes de sintetizar nada hay que conocer los datos reales. Este cuaderno describe los cuatro
estudios, su censura y sus curvas de Kaplan-Meier, que serán la referencia de todo lo demás.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from synapse import datasets, generators, inference as inf, utility as ut, privacy as pv, compliance, viz
from synapse import experiment as ex
viz.style(); pd.set_option("display.precision", 3); pd.set_option("display.width", 140)
TABLES = ROOT / "results" / "tables"
metrics = pd.read_csv(TABLES / "study_metrics.csv") if (TABLES / "study_metrics.csv").exists() else None

## Resumen de los cuatro estudios

In [2]:
rows = []
for name in datasets.LOADERS:
    d = datasets.load(name)
    ev = d.data["event"].astype(int)
    rows.append({"estudio": name, "n": d.n, "eventos": int(ev.sum()), "% censura": round(100 * (1 - ev.mean()), 1),
                 "mediana seguimiento (días)": float(d.data["time"].median()), "covariables": len(d.covariates),
                 "fuente": d.citation.split(".")[0]})
pd.DataFrame(rows)

,estudio,n,eventos,% censura,mediana seguimiento (días),covariables,fuente
0,gbsg2,686,299,56.4,1084.0,8,"Schumacher M, et al"
1,whas500,500,215,57.0,631.5,14,"Hosmer DW, Lemeshow S, May S"
2,flchain,6524,1962,69.9,4303.0,8,"Dispenzieri A, et al"
3,veterans,137,128,6.6,80.0,6,"Kalbfleisch JD, Prentice RL"


**Lectura.** *veterans* (n = 137, 93 % de eventos) es el caso extremo de muestra pequeña:
cualquier paciente es fácil de individualizar. *flchain* (n = 6.524, 70 % censurados) es el caso
grande y muy censurado; en el estudio se submuestrea a 2.000 para que los ataques sean asumibles.

## Curvas de Kaplan-Meier

In [3]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, name in zip(axes, datasets.LOADERS):
    d = datasets.load(name)
    inf.kaplan_meier(d.data, name).plot_survival_function(ax=ax, color=viz.REAL, ci_alpha=.15, show_censors=False)
    ax.set_title(name); ax.set_xlabel("Días"); ax.get_legend().remove()
axes[0].set_ylabel("S(t)"); plt.tight_layout()

## El análisis de referencia: Cox sobre los datos reales completos

In [4]:
d = datasets.load("gbsg2")
fit = inf.fit_cox(d.data, d)
fit.assign(HR=np.exp(fit["coef"]), HR_inf=np.exp(fit["lower"]), HR_sup=np.exp(fit["upper"]))[["HR", "HR_inf", "HR_sup", "p"]]

,HR,HR_inf,HR_sup,p
term,,,,
horTh[yes],0.707,0.549,0.911,7.301e-03
age,0.991,0.973,1.009,3.091e-01
menostat[Post],1.295,0.904,1.855,1.590e-01
tsize,1.008,1.000,1.016,4.779e-02
tgrade[II],1.889,1.159,3.079,1.069e-02
tgrade[III],2.181,1.288,3.691,3.685e-03
pnodes,1.050,1.035,1.065,5.699e-11
progrec,0.998,0.997,0.999,1.107e-04
estrec,1.000,0.999,1.001,6.613e-01


**Comprobación con la literatura.** La hormonoterapia (`horTh[yes]`) reduce el riesgo de
recidiva (HR ≈ 0,71; p = 0,007) y cada ganglio afectado lo aumenta, como en Schumacher et al. (1994).
Esta es la «verdad» que un buen sintético debería reproducir.

> 📝 **Tarea para Maricel:** añadir aquí una tabla 1 descriptiva (media ± DE o mediana [RIC] por
> variable) para cada estudio, en el formato habitual de una revista clínica.